# 1. 加载数据

## 1.1 加载数据

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
df = pd.read_csv('../data/train.csv')


FileNotFoundError: [Errno 2] No such file or directory: '../data/train.csv'

## 1.2. 划分数据集

In [26]:

X = df.drop('label', axis=1)
y = df['label']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)



# 1.3. 特征工程：归一化

| 方法 | 做什么                           | 适用场景 | 是否计算参数 | 是否会导致数据泄露 |
| :--- |:------------------------------| :--- | :--- | :--- |
| **`fit_transform(X)`** | 先 fit（学习参数），再 transform（应用转换） | **训练集** | ✅ 会 | ❌ 不会（用在训练集上安全） |
| **`transform(X)`** | 只用已有的参数做转换，不重新计算              | **测试集 / 新数据** | ❌ 不会 | ✅ 不会（用训练集参数，无泄露） |
fit 算出“参数”就是训练集每个特征的统计量（均值、标准差、最值等），transform 就是用这些统计量去转换数据。

In [27]:
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

 # 4. 转换成Tensor

In [28]:
x_train = torch.tensor(x_train).float()
x_test = torch.tensor(x_test).float()
y_train = torch.tensor(y_train.values)
y_test = torch.tensor(y_test.values)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

torch.Size([29400, 784]) torch.Size([12600, 784]) torch.Size([29400]) torch.Size([12600])


# 2. 定义模型

In [29]:
from torch import nn

model = nn.Sequential(
    nn.Linear(784, 50),
    nn.ReLU(),
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 10),
    nn.Softmax(dim=-1)
)

In [30]:

state_dict = torch.load("../data/nn_example.pt",map_location=torch.device('cpu'))
model.load_state_dict(state_dict)

<All keys matched successfully>

# 4. 预测

In [31]:

y_pred_proba = model(x_test)
print(y_pred_proba.shape)

y_pred_label = torch.argmax(y_pred_proba, dim=-1)
print(y_pred_label.shape)
print(y_pred_label[0:10])



torch.Size([12600, 10])
torch.Size([12600])
tensor([8, 1, 9, 9, 8, 6, 2, 2, 7, 1])


# 5. 模型评估：计算准确率

In [32]:
acc_cnt = (y_pred_label == y_test).sum().item()
print("预测准确率: ", acc_cnt / len(y_test))

预测准确率:  0.9681746031746031
